In [33]:
# =========================================================
# IMPORTS
# =========================================================

import numpy as np
import pandas as pd

import mlflow
import mlflow.xgboost
import mlflow.lightgbm

import optuna

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mlflow.set_experiment(
    "uber_demand_forecasting_final"
)

<Experiment: artifact_location='file:///c:/Users/Soham/Documents/UBER DEMAND PREDICATION/mlruns/13', creation_time=1779904673003, experiment_id='13', last_update_time=1779904673003, lifecycle_stage='active', name='uber_demand_forecasting_final', tags={}, trace_location=None, workspace='default'>

In [34]:
# load the training and test data

train_data_path = "data/train_new.csv"
test_data_path = "data/test_new.csv"

train_df = pd.read_csv(train_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

test_df = pd.read_csv(test_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

In [40]:
print((test_df["total_pickups"] != 0).sum())
print((train_df["total_pickups"] != 0).sum())

88501
169791


In [41]:
print((test_df["total_pickups"] == 0).sum())
print((train_df["total_pickups"] == 0).sum())

779
2889


In [36]:
import mlflow
import mlflow.xgboost
import optuna
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.model_selection import TimeSeriesSplit

# ===================================================
# CREATE TIME FEATURES FOR TRAIN DATA
# ===================================================

df = train_df.copy()

# ensure datetime index
df.index = pd.to_datetime(df.index)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

df["hour"] = df.index.hour

df["day"] = df.index.day

df["month"] = df.index.month

df["day_of_week"] = df.index.day_name()

df["is_weekend"] = (
    df.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# ===================================================
# EXTRA FEATURE ENGINEERING
# ===================================================

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

df["lag_1"] = (
    df["total_pickups"]
    .shift(1)
)

df["lag_24"] = (
    df["total_pickups"]
    .shift(24)
)

df["lag_168"] = (
    df["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

df["rolling_mean_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

df["rolling_std_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

df["rolling_mean_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .mean()
)

df["rolling_std_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .std()
)

# ---------------------------------------------------
# PEAK HOUR FEATURE
# ---------------------------------------------------

df["is_peak_hour"] = (
    df["hour"].isin(
        [7, 8, 9, 17, 18, 19]
    )
).astype(int)

# ---------------------------------------------------
# NIGHT FEATURE
# ---------------------------------------------------

df["is_night"] = (
    df["hour"].isin(
        [0, 1, 2, 3, 4, 5]
    )
).astype(int)

# ---------------------------------------------------
# INTERACTION FEATURE
# ---------------------------------------------------

df["weekend_hour_interaction"] = (
    df["hour"] *
    df["is_weekend"]
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

df = df.dropna()

# ===================================================
# CREATE TEST FEATURES
# ===================================================

test_processed = test_df.copy()

test_processed.index = pd.to_datetime(
    test_processed.index
)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

test_processed["hour"] = (
    test_processed.index.hour
)

test_processed["day"] = (
    test_processed.index.day
)

test_processed["month"] = (
    test_processed.index.month
)

test_processed["day_of_week"] = (
    test_processed.index.day_name()
)

test_processed["is_weekend"] = (
    test_processed.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

test_processed["hour_sin"] = np.sin(
    2 * np.pi * test_processed["hour"] / 24
)

test_processed["hour_cos"] = np.cos(
    2 * np.pi * test_processed["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

test_processed["lag_1"] = (
    test_processed["total_pickups"]
    .shift(1)
)

test_processed["lag_24"] = (
    test_processed["total_pickups"]
    .shift(24)
)

test_processed["lag_168"] = (
    test_processed["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

test_processed["rolling_mean_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

test_processed["rolling_std_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

test_processed["rolling_mean_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .mean()
)

test_processed["rolling_std_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .std()
)

# ---------------------------------------------------
# PEAK HOUR FEATURE
# ---------------------------------------------------

test_processed["is_peak_hour"] = (
    test_processed["hour"].isin(
        [7, 8, 9, 17, 18, 19]
    )
).astype(int)

# ---------------------------------------------------
# NIGHT FEATURE
# ---------------------------------------------------

test_processed["is_night"] = (
    test_processed["hour"].isin(
        [0, 1, 2, 3, 4, 5]
    )
).astype(int)

# ---------------------------------------------------
# INTERACTION FEATURE
# ---------------------------------------------------

test_processed["weekend_hour_interaction"] = (
    test_processed["hour"] *
    test_processed["is_weekend"]
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

test_processed = test_processed.dropna()

# ===================================================
# FEATURES AND TARGET
# ===================================================

X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df["total_pickups"]

X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed["total_pickups"]

# ===================================================
# CATEGORICAL COLUMNS
# ===================================================

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
    )

# ===================================================
# TIME SERIES CROSS VALIDATION
# ===================================================

tscv = TimeSeriesSplit(
    n_splits=5
)

# ===================================================
# MLFLOW EXPERIMENT
# ===================================================

mlflow.set_experiment(
    "xgboost_rolling_rmse_cv"
)

# ===================================================
# OBJECTIVE FUNCTION
# ===================================================

def objective(trial):

    rmse_scores = []

    with mlflow.start_run(nested=True):

        params = {

            "n_estimators": trial.suggest_int(
                "n_estimators",
                200,
                600,
                step=50
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                1e-2,
                0.1,
                log=True
            ),

            "max_depth": trial.suggest_int(
                "max_depth",
                3,
                8
            ),

            "subsample": trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),

            "min_child_weight": trial.suggest_int(
                "min_child_weight",
                2,
                10
            ),

            "gamma": trial.suggest_float(
                "gamma",
                0,
                3
            ),

            "reg_alpha": trial.suggest_float(
                "reg_alpha",
                1e-4,
                5,
                log=True
            ),

            "reg_lambda": trial.suggest_float(
                "reg_lambda",
                1e-4,
                5,
                log=True
            ),

            "eval_metric": "rmse",

            "enable_categorical": True,

            "tree_method": "hist",

            "random_state": 42,

            "n_jobs": -1
        }

        # ============================================
        # CROSS VALIDATION
        # ============================================

        for train_idx, val_idx in tscv.split(X_full):

            X_train = X_full.iloc[
                train_idx
            ]

            X_val = X_full.iloc[
                val_idx
            ]

            y_train = y_full.iloc[
                train_idx
            ]

            y_val = y_full.iloc[
                val_idx
            ]

            model = XGBRegressor(
                **params
            )

            model.fit(

                X_train,
                y_train,

                eval_set=[
                    (X_val, y_val)
                ],

                verbose=False
            )

            y_pred = model.predict(
                X_val
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y_val,
                    y_pred
                )
            )

            rmse_scores.append(
                rmse
            )

        # ============================================
        # MEAN CV RMSE
        # ============================================

        mean_rmse = np.mean(
            rmse_scores
        )

        mlflow.log_params(
            params
        )

        mlflow.log_metric(
            "CV_RMSE",
            mean_rmse
        )

        return mean_rmse

# ===================================================
# OPTUNA SEARCH
# ===================================================

with mlflow.start_run(

    run_name="xgboost_rmse_cv_tuning"

):

    study = optuna.create_study(
        direction="minimize"
    )

    study.optimize(

        objective,

        n_trials=50,

        n_jobs=1
    )

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "best_cv_rmse",
        study.best_value
    )

print("Best Params:")
print(study.best_params)

print("\nBest CV RMSE:")
print(study.best_value)

# ===================================================
# FINAL MODEL
# ===================================================

best_params = study.best_params

final_model = XGBRegressor(

    **best_params,

    eval_metric="rmse",

    enable_categorical=True,

    tree_method="hist",

    random_state=42,

    n_jobs=-1
)

# ===================================================
# TRAIN FINAL MODEL
# ===================================================

final_model.fit(

    X_full,
    y_full,

    verbose=False
)

# ===================================================
# TEST PREDICTIONS
# ===================================================

y_test_pred = final_model.predict(
    X_test
)

# ===================================================
# FINAL METRICS
# ===================================================

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_mape = mean_absolute_percentage_error(
    y_test,
    y_test_pred
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

# ===================================================
# PRINT RESULTS
# ===================================================

print("\nFINAL TEST RESULTS")

print("MAE :", final_mae)

print("RMSE:", final_rmse)

print("MAPE:", final_mape)

print("R2  :", final_r2)

# ===================================================
# LOG FINAL MODEL
# ===================================================

with mlflow.start_run(

    run_name="best_xgboost_rmse_cv_model"

):

    mlflow.log_params(
        best_params
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_MAPE",
        final_mape
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.xgboost.log_model(
        xgb_model=final_model,
        name="model"
    )

print(
    "\nFinal XGBoost model logged successfully."
)

2026/05/28 04:25:39 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_rolling_rmse_cv' does not exist. Creating a new experiment.
[I 2026-05-28 04:25:39,533] A new study created in memory with name: no-name-b40b95de-2155-4e22-b234-52eb8e019528
[I 2026-05-28 04:25:50,519] Trial 0 finished with value: 28.99151850233606 and parameters: {'n_estimators': 350, 'learning_rate': 0.014660163691036843, 'max_depth': 4, 'subsample': 0.7241064107237593, 'colsample_bytree': 0.7456388768452629, 'min_child_weight': 8, 'gamma': 2.3170299242990575, 'reg_alpha': 0.023563132927825702, 'reg_lambda': 0.0007379448977224283}. Best is trial 0 with value: 28.99151850233606.
[I 2026-05-28 04:26:21,990] Trial 1 finished with value: 31.89275828111762 and parameters: {'n_estimators': 450, 'learning_rate': 0.023324358277094608, 'max_depth': 8, 'subsample': 0.7438327358033594, 'colsample_bytree': 0.9748703836680006, 'min_child_weight': 6, 'gamma': 1.7408500042556434, 'reg_alpha': 1.7301403562765616, 'reg_lam

Best Params:
{'n_estimators': 400, 'learning_rate': 0.014018132043921501, 'max_depth': 4, 'subsample': 0.9517761136340085, 'colsample_bytree': 0.7312811935505404, 'min_child_weight': 6, 'gamma': 2.9875347120178355, 'reg_alpha': 0.010117602216555105, 'reg_lambda': 0.00023672817660639283}

Best CV RMSE:
28.79335934612987

FINAL TEST RESULTS
MAE : 13.556848526000977
RMSE: 22.004855227073254
MAPE: 393281833795584.0
R2  : 0.971659243106842

Final XGBoost model logged successfully.


In [42]:
# ===================================================
# WAPE
# ===================================================

final_wape = (
    np.sum(
        np.abs(y_test - y_test_pred)
    )
    /
    np.sum(
        np.abs(y_test)
    )
) * 100

print("WAPE:", final_wape)

WAPE: 10.275203447713388


In [43]:
import mlflow
import mlflow.xgboost
import optuna
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.model_selection import TimeSeriesSplit

# ===================================================
# CREATE TIME FEATURES FOR TRAIN DATA
# ===================================================

df = train_df.copy()

# ensure datetime index
df.index = pd.to_datetime(df.index)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

df["hour"] = df.index.hour

df["day"] = df.index.day

df["month"] = df.index.month

df["day_of_week"] = df.index.day_name()

df["is_weekend"] = (
    df.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# ===================================================
# EXTRA FEATURE ENGINEERING
# ===================================================

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

df["lag_1"] = (
    df["total_pickups"]
    .shift(1)
)

df["lag_24"] = (
    df["total_pickups"]
    .shift(24)
)

df["lag_168"] = (
    df["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

df["rolling_mean_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

df["rolling_std_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

df["rolling_mean_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .mean()
)

df["rolling_std_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .std()
)

# ---------------------------------------------------
# PEAK HOUR FEATURE
# ---------------------------------------------------

df["is_peak_hour"] = (
    df["hour"].isin(
        [7, 8, 9, 17, 18, 19]
    )
).astype(int)

# ---------------------------------------------------
# NIGHT FEATURE
# ---------------------------------------------------

df["is_night"] = (
    df["hour"].isin(
        [0, 1, 2, 3, 4, 5]
    )
).astype(int)

# ---------------------------------------------------
# INTERACTION FEATURE
# ---------------------------------------------------

df["weekend_hour_interaction"] = (
    df["hour"] *
    df["is_weekend"]
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

df = df.dropna()

# ===================================================
# CREATE TEST FEATURES
# ===================================================

test_processed = test_df.copy()

test_processed.index = pd.to_datetime(
    test_processed.index
)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

test_processed["hour"] = (
    test_processed.index.hour
)

test_processed["day"] = (
    test_processed.index.day
)

test_processed["month"] = (
    test_processed.index.month
)

test_processed["day_of_week"] = (
    test_processed.index.day_name()
)

test_processed["is_weekend"] = (
    test_processed.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

test_processed["hour_sin"] = np.sin(
    2 * np.pi * test_processed["hour"] / 24
)

test_processed["hour_cos"] = np.cos(
    2 * np.pi * test_processed["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

test_processed["lag_1"] = (
    test_processed["total_pickups"]
    .shift(1)
)

test_processed["lag_24"] = (
    test_processed["total_pickups"]
    .shift(24)
)

test_processed["lag_168"] = (
    test_processed["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

test_processed["rolling_mean_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

test_processed["rolling_std_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

test_processed["rolling_mean_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .mean()
)

test_processed["rolling_std_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .std()
)

# ---------------------------------------------------
# PEAK HOUR FEATURE
# ---------------------------------------------------

test_processed["is_peak_hour"] = (
    test_processed["hour"].isin(
        [7, 8, 9, 17, 18, 19]
    )
).astype(int)

# ---------------------------------------------------
# NIGHT FEATURE
# ---------------------------------------------------

test_processed["is_night"] = (
    test_processed["hour"].isin(
        [0, 1, 2, 3, 4, 5]
    )
).astype(int)

# ---------------------------------------------------
# INTERACTION FEATURE
# ---------------------------------------------------

test_processed["weekend_hour_interaction"] = (
    test_processed["hour"] *
    test_processed["is_weekend"]
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

test_processed = test_processed.dropna()

# ===================================================
# FEATURES AND TARGET
# ===================================================

X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df["total_pickups"]

X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed["total_pickups"]

# ===================================================
# CATEGORICAL COLUMNS
# ===================================================

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
    )

# ===================================================
# WAPE FUNCTION
# ===================================================

def wape(y_true, y_pred):

    return (
        np.sum(
            np.abs(y_true - y_pred)
        )
        /
        np.sum(
            np.abs(y_true)
        )
    ) * 100

# ===================================================
# TIME SERIES CROSS VALIDATION
# ===================================================

tscv = TimeSeriesSplit(
    n_splits=5
)

# ===================================================
# MLFLOW EXPERIMENT
# ===================================================

mlflow.set_experiment(
    "xgboost_mae_cv_rolling"
)

# ===================================================
# OBJECTIVE FUNCTION
# ===================================================

def objective(trial):

    mae_scores = []

    with mlflow.start_run(nested=True):

        params = {

            "n_estimators": trial.suggest_int(
                "n_estimators",
                200,
                600,
                step=50
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                1e-2,
                0.1,
                log=True
            ),

            "max_depth": trial.suggest_int(
                "max_depth",
                3,
                8
            ),

            "subsample": trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),

            "min_child_weight": trial.suggest_int(
                "min_child_weight",
                2,
                10
            ),

            "gamma": trial.suggest_float(
                "gamma",
                0,
                3
            ),

            "reg_alpha": trial.suggest_float(
                "reg_alpha",
                1e-4,
                5,
                log=True
            ),

            "reg_lambda": trial.suggest_float(
                "reg_lambda",
                1e-4,
                5,
                log=True
            ),

            "eval_metric": "mae",

            "enable_categorical": True,

            "tree_method": "hist",

            "random_state": 42,

            "n_jobs": -1
        }

        # ============================================
        # CROSS VALIDATION
        # ============================================

        for train_idx, val_idx in tscv.split(X_full):

            X_train = X_full.iloc[
                train_idx
            ]

            X_val = X_full.iloc[
                val_idx
            ]

            y_train = y_full.iloc[
                train_idx
            ]

            y_val = y_full.iloc[
                val_idx
            ]

            model = XGBRegressor(
                **params
            )

            model.fit(

                X_train,
                y_train,

                eval_set=[
                    (X_val, y_val)
                ],

                verbose=False
            )

            y_pred = model.predict(
                X_val
            )

            mae = mean_absolute_error(
                y_val,
                y_pred
            )

            mae_scores.append(
                mae
            )

        # ============================================
        # MEAN CV MAE
        # ============================================

        mean_mae = np.mean(
            mae_scores
        )

        mlflow.log_params(
            params
        )

        mlflow.log_metric(
            "CV_MAE",
            mean_mae
        )

        return mean_mae

# ===================================================
# OPTUNA SEARCH
# ===================================================

with mlflow.start_run(

    run_name="xgboost_mae_cv_tuning"

):

    study = optuna.create_study(
        direction="minimize"
    )

    study.optimize(

        objective,

        n_trials=50,

        n_jobs=1
    )

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "best_cv_mae",
        study.best_value
    )

print("Best Params:")
print(study.best_params)

print("\nBest CV MAE:")
print(study.best_value)

# ===================================================
# FINAL MODEL
# ===================================================

best_params = study.best_params

final_model = XGBRegressor(

    **best_params,

    eval_metric="mae",

    enable_categorical=True,

    tree_method="hist",

    random_state=42,

    n_jobs=-1
)

# ===================================================
# TRAIN FINAL MODEL
# ===================================================

final_model.fit(

    X_full,
    y_full,

    verbose=False
)

# ===================================================
# TEST PREDICTIONS
# ===================================================

y_test_pred = final_model.predict(
    X_test
)

# ===================================================
# FINAL METRICS
# ===================================================

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_mape = mean_absolute_percentage_error(
    y_test,
    y_test_pred
)

final_wape = wape(
    y_test,
    y_test_pred
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

# ===================================================
# PRINT RESULTS
# ===================================================

print("\nFINAL TEST RESULTS")

print("MAE :", final_mae)

print("RMSE:", final_rmse)

print("MAPE:", final_mape)

print("WAPE:", final_wape)

print("R2  :", final_r2)

# ===================================================
# LOG FINAL MODEL
# ===================================================

with mlflow.start_run(

    run_name="best_xgboost_mae_cv_model"

):

    mlflow.log_params(
        best_params
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_MAPE",
        final_mape
    )

    mlflow.log_metric(
        "FINAL_WAPE",
        final_wape
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.xgboost.log_model(
        xgb_model=final_model,
        name="model"
    )

print(
    "\nFinal XGBoost model logged successfully."
)

2026/05/28 05:01:32 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_mae_cv_rolling' does not exist. Creating a new experiment.
[I 2026-05-28 05:01:32,360] A new study created in memory with name: no-name-e9942e0f-c7c3-427f-88ea-20a76531810a
[I 2026-05-28 05:01:41,695] Trial 0 finished with value: 17.06343193054199 and parameters: {'n_estimators': 300, 'learning_rate': 0.012638672147760148, 'max_depth': 5, 'subsample': 0.7390432816702104, 'colsample_bytree': 0.916243607547931, 'min_child_weight': 5, 'gamma': 1.4286552931529302, 'reg_alpha': 4.846362118894925, 'reg_lambda': 0.0020020866238162644}. Best is trial 0 with value: 17.06343193054199.
[I 2026-05-28 05:01:48,236] Trial 1 finished with value: 18.790868949890136 and parameters: {'n_estimators': 250, 'learning_rate': 0.061304436766180165, 'max_depth': 5, 'subsample': 0.7958638317722093, 'colsample_bytree': 0.9343255797172276, 'min_child_weight': 9, 'gamma': 0.9424106258035628, 'reg_alpha': 0.36518564212122157, 'reg_lambda

Best Params:
{'n_estimators': 250, 'learning_rate': 0.021361556388724796, 'max_depth': 4, 'subsample': 0.9296516549428425, 'colsample_bytree': 0.8437136809833924, 'min_child_weight': 6, 'gamma': 1.9388168222839934, 'reg_alpha': 0.0025076641643753087, 'reg_lambda': 0.044323930330342126}

Best CV MAE:
16.507982444763183

FINAL TEST RESULTS
MAE : 13.632781982421875
RMSE: 22.11844999266732
MAPE: 391470464892928.0
WAPE: 10.33275577150634
R2  : 0.9713658690452576

Final XGBoost model logged successfully.
